# Conductivity from fixed thermometers

We infer every conductivity node from $-\nabla\cdot(h\nabla T)=q$ on the unit square, with a 300 K bottom wall and insulated remaining walls. This is an **illustrative, unqualified calculation**. A completed notebook establishes execution; chain convergence, heat-solve accuracy, mesh error and prior adequacy each need separate evidence. The planned scientific study is `docs/validation/hmc-mesh-study.md`; it has no results yet.

The 256 interior thermometers stay at the same physical positions when the inference mesh changes. Data come from a finer mesh and one fixed noise realization. The independent known error is $\sigma=10^{-3}$ K, rather than a spatially integrated misfit.


In [ ]:
nmesh = 16
fine_nmesh = 64
chains = 4
warmup = 100
draws = 200
seed = 7
sigma = 1e-3
output = None  # unique run directory under runs/
plot = True


The prior is illustrative: $m=\log(h/h_{ref})=\mu+\tau^{-1}(\kappa^2M+K)^{-1}M_L^{1/2}z$, with $z\sim N(0,I)$, $\mu=0$, $\kappa=2$, $\tau=1$, $h_{ref}=1$. Its boundaries are natural Neumann boundaries. These parameters remain fixed; they are not inferred. The inverse correlation scale uses the physical coordinate units, and $h_{ref}$ has conductivity units. These choices are not a mesh-accuracy or material-prior qualification.

Run with the dedicated Python 3.12+ CPU environment described in `docs/installation.md`. Restart the kernel after setting the JAX environment if it was already imported. Flat imports below must resolve to this checkout.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np

os.environ.setdefault("JAX_ENABLE_X64", "true")
os.environ.setdefault("JAX_PLATFORMS", "cpu")
# Running from either the repository or its notebooks/ directory uses this source.
root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
assert (root / "src/heat_inference.py").is_file()
sys.path.insert(0, str(root / "src"))

from forward_solver import SteadyHeat2DForwardSolver
from temperature_observations import TemperatureObservations, SensorOperator
from conductivity_prior import GaussianLogConductivity
from heat_model import FrozenHeatModel, HeatNumerics, NodalField
from heat_inference import HeatInverseProblem
from hmc_blackjax import HMCControls, require_sampling_environment

print(require_sampling_environment())


In [ ]:
def true_conductivity(x):
    return 1.0 + 0.2 * x[0] + 0.1 * x[1]

assert fine_nmesh > nmesh, "Synthetic data must use a finer mesh."
truth = SteadyHeat2DForwardSolver(
    nmesh=fine_nmesh, mesh_type="triangle", h=true_conductivity, q=1.0
)
truth.solve()
axis = np.arange(1, 17) / 17.0
sensor_points = np.array([(x, y) for x in axis for y in axis])
assert np.all((sensor_points > 0) & (sensor_points < 1))
exact_readings = SensorOperator(truth.V, sensor_points).predict(truth.T.x.array)
data = exact_readings + np.random.default_rng(seed).normal(0, sigma, len(sensor_points))
observations = TemperatureObservations(sensor_points, data, sigma)

forward = SteadyHeat2DForwardSolver(
    nmesh=nmesh, mesh_type="triangle", h=1.0, q=1.0
)
forward.solve()
before_h = np.array(forward.h.function.value, copy=True)
before_T = forward.T.x.array.copy()
problem = HeatInverseProblem(
    forward=forward, observations=observations,
    prior=GaussianLogConductivity(mu=0, kappa=2, tau=1, h_ref=1),
    numerics=HeatNumerics(),
)
assert observations.points.shape == (256, 2)


Scalar, callable and coordinate-labelled nodal conductivities specify an **initial field**, independently of the prior. Nodal inputs must match the inference mesh; there is no automatic cross-mesh field transfer. Optimization finds the mode of the density in log conductivity and reports its corresponding physical field. Hand the whole estimate to sampling so that every inferred coordinate survives.


In [ ]:
scalar_start = problem.initial_state(1.0)
callable_start = problem.initial_state(lambda x: 1.0 + 0.05 * x[0])
coordinates = FrozenHeatModel.from_forward(forward).coordinates
nodal_start = problem.initial_state(NodalField(np.ones(len(coordinates)), coordinates))
estimate = problem.optimize(initial_h=1.0)
print("Log-conductivity mode search:", estimate.success, estimate.message)
# Even a failed optimizer has an explicitly labelled last finite complete state.
# Notebook-only spacing: the measured nmesh=4 local curvature is stiff.
# Keep the library's randomized duration law and step-size-only adaptation.
controls = HMCControls(
    initial_step_size=5e-4, step_size_bounds=(1e-6, 1e-3)
)
samples = problem.sample(
    initial=estimate, seed=seed, chains=chains, warmup=warmup, draws=draws,
    output=output, controls=controls,
)
assert np.array_equal(before_h, forward.h.function.value)
assert np.array_equal(before_T, forward.T.x.array)
print("Saved calculation:", samples.output)


Four chains are the default. A common state is dispersed in whitened coordinates, with the reference-informed default `HMCControls(initial_spread=1.0)`; the library also starts at spacing `6.5e-4` with bounds `(1e-6,1.3e-3)`. These defaults are conditional on the documented reference experiment. To supply exact per-chain starts use `initial_states=[scalar_start, callable_start, nodal_start, estimate.state]` together with `chains=4` and omit `initial`. A single-chain execution check uses `chains=1` and cannot establish the required four-chain convergence.

The notebook uses an explicit initial spacing `5e-4` and upper bound `1e-3`, with the library's randomized duration interval `(np.pi/4, 3*np.pi/4)` and target acceptance `0.8`. A measured nmesh=4 probe found largest local curvature $1.77787\times10^6$, giving a local harmonic spacing scale $2/\sqrt{\lambda_{max}}=0.00149996$. The ceiling is below that local scale; this does not establish global stability. The former provisional library spacing `0.1` produced two rejected divergent warmup trajectories, followed by a fatal adjoint failure at an extreme positive finite conductivity. Those stopped records were retained as diagnostic evidence. The library now uses the reference-informed spacing `6.5e-4`, ceiling `1.3e-3` and spread `1.0`; this notebook retains its independently measured explicit controls. Only spacing adapts in warmup. The likelihood supplies kicks, the Gaussian prior supplies exact rotations, and the momentum covariance is identity. Retained iterations use frozen controls; every rejection repeat is saved. Each physical study still requires numerical and convergence qualification.


In [ ]:
# Diagnose first: summary() does not compute convergence diagnostics itself.
report = samples.diagnostics()
summary = samples.summary()
print("Chain convergence qualified:", report.qualified)
print("Reasons:", report.reasons)
print("Retained states per chain:", summary.counts.tolist())
# Physical h is transformed before averaging or taking quantiles.
assert summary.mean.shape == ((nmesh + 1) ** 2,)
assert summary.counts.tolist() == [draws] * chains
assert summary.qualified == report.qualified
selected_h = samples.field(chain=0, draw=0)
selected_state = samples.state(chain=0, draw=0)
selected_T = samples.temperature(chain=0, draw=0)
predictions = np.concatenate([
    samples.sensor_predictions(chain=c) for c in range(chains)
], axis=0)
mean_prediction = predictions.mean(axis=0)
standardized_residual = (mean_prediction - data) / sigma
# Full loading is explicit; use field() or load_draws(start=..., stop=...) for subsets.
small_subset = samples.load_draws(chains=[0], start=0, stop=min(2, draws))

# Export small arrays for a separate plotting environment as well.
np.savez(
    samples.output / "illustrative-fields.npz", coordinates=summary.coordinates,
    cells=summary.cells, mean=summary.mean, sd=summary.sd,
    quantiles=summary.quantiles, selected_T=selected_T,
    sensor_points=sensor_points, data=data, sigma=sigma,
    mean_prediction=mean_prediction, standardized_residual=standardized_residual,
)


The 5%–95% intervals are **pointwise** credible intervals, rather than a simultaneous band for the field. The means and quantiles are of physical $h$, so $E[h]\ne h_{ref}\exp(E[m])$ in general. Similarly $T(E[h])\ne E[T(h)]$: average `temperature(chain=..., draw=...)` fields to estimate mean temperature, or average the saved sensor predictions as above. Temperature uncertainty here describes latent heat predictions; a future noisy measurement also includes its measurement noise.

Matplotlib uses the recorded $(x,y)$ coordinates and triangle connectivity directly. The PyVista/VTK extra has a known conflict with the present FEM environment. Keep that environment intact. The optional existing plot route is `f=samples.as_function(summary="mean")` followed by `plot_scalar_mesh(f.function_space, f.x.array, "h")` in a separately qualified environment. Its physical-node association has an independent triangle/quadrilateral test.


In [ ]:
if plot:
    import matplotlib.pyplot as plt
    from matplotlib.tri import Triangulation

    xy = summary.coordinates
    triangulation = Triangulation(xy[:, 0], xy[:, 1], triangles=summary.cells)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), constrained_layout=True)
    for ax, values, title in zip(
        axes,
        [summary.mean, summary.sd, summary.quantiles[2] - summary.quantiles[0]],
        ["Mean conductivity", "Conductivity standard deviation", "Pointwise 90% interval width"],
    ):
        field = ax.tripcolor(triangulation, values, shading="gouraud")
        fig.colorbar(field, ax=ax)
        ax.set(xlabel="x", ylabel="y", title=title, aspect="equal")
    fig.suptitle("Illustrative posterior; inspect convergence report")
    fig.savefig(samples.output / "conductivity.png", dpi=120)
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(8, 3.2), constrained_layout=True)
    field = axes[0].tripcolor(triangulation, selected_T, shading="gouraud")
    fig.colorbar(field, ax=axes[0], label="K")
    axes[0].set(xlabel="x", ylabel="y", title="Temperature for retained field 0", aspect="equal")
    field = axes[1].scatter(sensor_points[:, 0], sensor_points[:, 1], c=standardized_residual)
    fig.colorbar(field, ax=axes[1], label="(mean predicted T - observed T) / sigma")
    axes[1].set(xlabel="x", ylabel="y", title="Thermometer residuals", aspect="equal")
    fig.savefig(samples.output / "temperature-residuals.png", dpi=120)
    plt.show()


Each run has `problem.json`, `problem.npz`, `chains.sqlite` and advisory `status.json`; diagnostics add `diagnostics.json`. Keep long histories in run directories. The SQLite record holds every warmup/retained iteration and the random/continuation state. Copy a closed run directory, or use SQLite backup; copying an active database alone is not a backup.

After interruption, load the saved experiment and call `resume(output)` to finish its original per-chain target. Explicitly extend with `resume(output, total_draws=larger_total)`; this invalidates old convergence qualification. Changed physics, numerical tolerances, source or environment cannot append to the same calculation. The following idempotent resume changes no retained history because this run has finished.


In [ ]:
with HeatInverseProblem.load(samples.output) as restored_problem:
    assert restored_problem.to_record().problem_id == problem.to_record().problem_id
    with restored_problem.resume(samples.output) as restored:
        assert restored.summary().counts.tolist() == [draws] * chains

# Every operation above preserved the caller's physical fields.
assert np.array_equal(before_h, forward.h.function.value)
assert np.array_equal(before_T, forward.T.x.array)
# Explicit physical application is the sole requested mutation of caller h and T.
T_at_mean_h = samples.apply_to(forward, summary="mean")
assert T_at_mean_h is forward.T
print("Applied mean conductivity; resulting temperature is T(mean(h)).")
samples.close()
problem.close()


Convergence qualification requires four chains, rank-normalized split/folded $\widehat R<1.01$, bulk and tail ESS at least 400 for every monitored nonconstant quantity, and zero retained divergences. Inspect Monte Carlo errors in `report.per_quantity` and worst physical locations in `report.worst_locations`. Short runs, missing optional diagnostics and nonfinite diagnostics cannot qualify. For prescribed boundary thermometers, zero stochastic uncertainty is separate from interpolation consistency: `report.fixed_sensor_consistency` preserves raw residuals and labels every nonzero discrepancy unverified. This notebook uses interior sensors.

CPU calculations use one MPI rank per heat solve. Existing integrated-misfit TAO reconstruction remains available with its original mathematics. The posterior calculation has a sensor likelihood and a specified Gaussian log-field prior; it defines its own inference problem.


## Compare meshes using one frozen physical observation record

The full smooth/sharp study is pending. Generate fine-truth observations with the
`truth` command, qualify a numerical pilot using that actual data record, and run
the six baseline plus eight predeclared n16 sensitivity calculations described in
`docs/validation/hmc-mesh-study.md`. The cells below read completed evidence only.
An absent result stays explicitly absent. MCSE concerns finite sampling; changes
across meshes or priors compare different posterior laws. Spatial inclusion near
the interface is one-field/noise evidence, not repeated-noise coverage.


In [ ]:
from pathlib import Path
import json
from IPython.display import display, Image
from hmc_validation import write_study_report

mesh_root = Path("../runs")
mesh_manifests = []
for candidate in sorted(mesh_root.glob("*/manifest.json")):
    identity = json.loads(candidate.read_text())["identity"]
    if identity["kind"] == "mesh":
        mesh_manifests.append(candidate)
write_study_report(mesh_manifests, mesh_root / "hmc-mesh-report.md")
print(f"Recorded mesh/prior attempts: {len(mesh_manifests)}; 14 outcomes required.")
for manifest in mesh_manifests:
    result_path = manifest.parent / "result.json"
    if not result_path.exists():
        print(manifest.parent.name, "incomplete")
        continue
    evidence = json.loads(result_path.read_text())
    print(manifest.parent.name, evidence["status"])
    if evidence["status"] == "sampled":
        print("Spatial inclusion:", evidence["coverage"])
        print("MCSE at shared locations:", evidence["shared_locations"])
        print("Cost uncertainty:", evidence["uncertainty"]["status"])
        display(Image(filename=str(manifest.parent / "conductivity.png")))
        display(Image(filename=str(manifest.parent / "predictive-residuals.png")))
